NAV Domain — Raw Asynchronous Log Evidence
NAV Domain — Deterministic Time Reconstruction

In [19]:
import src.dashboard.nav.nav_data_service as nds
print(f"📍 Python is actually loading from: {nds.__file__}")

📍 Python is actually loading from: /home/ni/ardupilot-nav-domain-poc/src/dashboard/nav/nav_data_service.py


In [ ]:
import pandas as pd
from src.utils.db_connector import get_connection

# WE DEFINE IT HERE TO BYPASS THE DISK IMPORT ISSUE
class NavDataService:
    def __init__(self, mission_id: str):
        self.mission_id = mission_id
        self.available_views = self._discover_schema()
        print(f"🔎 Service initialized. Available Views: {self.available_views}")

    def _discover_schema(self):
        with get_connection() as con:
            res = con.execute("SELECT table_name FROM information_schema.views").fetchall()
            return [r[0] for r in res]

    def fetch_navigation_state(self) -> pd.DataFrame:
        query = """
        SELECT
            TimeUS / 1000000.0 AS mission_time,
            Lat / 10000000.0 AS lat,
            Lng / 10000000.0 AS lng,
            Alt AS alt,
            spd_gps AS speed,
            Roll, Pitch, Yaw
        FROM fact_nav_state_vector
        WHERE mission_id = ?
        ORDER BY TimeUS ASC
        """
        with get_connection() as con:
            return con.execute(query, [self.mission_id]).df()

# Initialize the service using THIS local definition
nav_service = NavDataService("manual_run")
print(f"✅ SUCCESS: Attribute available_views exists: {hasattr(nav_service, 'available_views')}")

🔎 Service initialized. Available Views: ['view_clean_est', 'view_clean_nav', 'character_sets', 'check_constraints', 'columns', 'constraint_column_usage', 'constraint_table_usage', 'key_column_usage', 'referential_constraints', 'schemata', 'tables', 'table_constraints', 'views', 'duckdb_columns', 'duckdb_constraints', 'duckdb_databases', 'duckdb_indexes', 'duckdb_logs', 'duckdb_schemas', 'duckdb_tables', 'duckdb_types', 'duckdb_views', 'pragma_database_list', 'sqlite_master', 'sqlite_schema', 'sqlite_temp_master', 'sqlite_temp_schema', 'pg_am', 'pg_attrdef', 'pg_attribute', 'pg_class', 'pg_constraint', 'pg_database', 'pg_depend', 'pg_description', 'pg_enum', 'pg_index', 'pg_indexes', 'pg_namespace', 'pg_prepared_statements', 'pg_proc', 'pg_sequence', 'pg_sequences', 'pg_settings', 'pg_tables', 'pg_tablespace', 'pg_type', 'pg_views']
✅ SUCCESS: Attribute available_views exists: True


In [ ]:
# 1. Define the class (The "Manual Override")
class NavDataService:
    def __init__(self, mission_id: str):
        self.mission_id = mission_id
        self.available_views = self._discover_schema()

    def _discover_schema(self):
        with get_connection() as con:
            res = con.execute("SELECT table_name FROM information_schema.views").fetchall()
            return [r[0] for r in res]

    def fetch_navigation_state(self):
        query = "SELECT * FROM fact_nav_state_vector WHERE mission_id = ?"
        with get_connection() as con:
            return con.execute(query, [self.mission_id]).df()

# 2. Define the Audit function in the SAME SCOPE
def run_audit_local(mission_id):
    # We explicitly use the class defined right above
    service = NavDataService(mission_id)

    # Check if the method exists before calling to debug
    if not hasattr(service, 'fetch_navigation_state'):
        return print("❌ Still seeing the OLD class version!")

    df = service.fetch_navigation_state()
    print(f"✅ Audit Success: {len(df)} rows found for {mission_id}")

# 3. Execute
run_audit_local("manual_run")

✅ Audit Success: 3830 rows found for manual_run
